# EADRO — training di Google Colab (GPU T4)

Notebook ini hanya untuk **training**. Data chunk dibangun di laptop (`pipeline/build_T.sh`, butuh Python 3.8 + `tick`), lalu dikemas dengan `pipeline/pack_for_colab.sh` dan diunggah ke Drive.

**Sebelum mulai**
1. Di laptop: `pipeline/pack_for_colab.sh A1` (atau `A1_T40`, dst.). Unggah semua file di `work/colab_pack/` ke satu folder Drive, mis. `MyDrive/eadro/`.
2. Di Colab: *Runtime → Change runtime type → T4 GPU*.
3. Jalankan sel dari atas ke bawah. **Jangan** lompat ke sel grid sebelum sel uji 1 epoch berhasil.

**Catatan keabsahan penelitian**: semua varian fusion yang dibandingkan harus dilatih di hardware yang sama, dan angka efisiensi (`infer_ms_per_window`, `peak_rss_mb`) tidak boleh dibandingkan lintas hardware.

## 1. Cek GPU dan Python
Harus muncul `Tesla T4` (atau GPU lain). Python 3.12 ideal, karena versi paket di bawah hanya diuji di 3.12.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

## 2. Hubungkan Drive dan atur konfigurasi
Ubah `DRIVE_DIR` dan `DATA_NAMES` sesuai yang Anda unggah. Hasil training ditulis ke `DRIVE_DIR/result` supaya tidak hilang saat sesi putus.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/eadro'   # folder berisi eadro_code.tar, eadro_data_<NAME>.tar, SHA256SUMS_<NAME>.txt
DATA_NAMES = ['A1']                           # data yang diunggah, mis. ['A1', 'A1_T40']
ROOT       = '/content/eadro'                 # tempat kerja di disk lokal Colab (jauh lebih cepat daripada Drive)
DATA_STR   = ' '.join(DATA_NAMES)

## 3. Ekstrak kode dan data, lalu verifikasi data
`sha256sum -c` membandingkan sidik jari tiap file chunk dengan yang dicatat di laptop. Semua baris harus `OK`. Ini bukti bahwa data di Colab **sama persis** dengan data di laptop. Simpan keluarannya untuk laporan.

In [ ]:
%%bash -s "$ROOT" "$DRIVE_DIR" "$DATA_STR"
set -euo pipefail
ROOT="$1"; DRIVE="$2"; NAMES="$3"
mkdir -p "$ROOT" && cd "$ROOT"
tar -xf "$DRIVE/eadro_code.tar"
for n in $NAMES; do
    echo "== $n =="
    tar -xf "$DRIVE/eadro_data_$n.tar"
    sha256sum -c "$DRIVE/SHA256SUMS_$n.txt"
done
mkdir -p work/preprocess work/result   # run_train.sh melakukan cd ke work/preprocess
du -sh work/chunks/*

## 4. Pasang environment training (±3–6 menit)
Membuat venv terpisah di `/content/venv` dengan torch 2.2.1 (CUDA 12.1) dan DGL 2.1.0, sama seperti env lokal Anda.

**Skrip ini berhenti dengan pesan `GAGAL:` jika DGL tidak bisa memakai GPU.** Penyebab yang mungkin: `data.dgl.ai` tidak terjangkau sehingga pip diam-diam memasang DGL versi CPU. Jangan lanjut ke sel berikutnya jika ini terjadi; kabari saya dengan pesan lengkapnya.

Jika Python di sel 1 bukan 3.12, coba (belum teruji): `!pip -q install uv && uv python install 3.12`, lalu ganti sel di bawah dengan `PYTHON_BIN=$(uv python find 3.12)` di depan perintah.

In [ ]:
%%bash -s "$ROOT"
set -euo pipefail
cd "$1"
TORCH_FLAVOR=cu121 bash pipeline/setup_train_env.sh /content/venv

## 5. Uji 1 epoch (wajib berhasil sebelum grid)
Dua hal yang diperiksa:
- Baris `Using GPU...` muncul (bukan `Using CPU...`).
- Waktu `Epoch 1/1 ... [xx.xxs]`. Di laptop (CPU) satu epoch data A1 ±64 detik. Perkiraan saya untuk T4 ±5–15 detik, tapi itu belum terukur; angka dari sel ini adalah yang sebenarnya.

In [ ]:
%%bash -s "$ROOT" "$DRIVE_DIR"
cd "$1"
export TRAIN_PY=/content/venv/bin/python RESULT_DIR="$2/result" DATA_NAME=A1
bash pipeline/run_train.sh within_event 42 1 2>&1 | grep -E "Using|Epoch|Best result|Efficiency|Traceback|Error"

## 6. Jalankan grid
Atur parameter, lalu jalankan. `run_grid.sh` mencatat run yang selesai di `DRIVE_DIR/result/grid_selesai.txt` dan **melewatinya** jika sel ini dijalankan ulang, jadi aman setelah sesi putus (run yang sedang berjalan saat putus diulang dari awal).

**Uji kesetaraan CPU vs GPU (dianjurkan dulu)**: pilih konfigurasi yang sudah punya 3 seed di CPU (mis. `within_event`, 15 epoch, `concat_glu`, seed `42 7 123`), jalankan di sini, lalu bandingkan dengan hasil CPU lama. Jika selisihnya masih dalam sebaran antar-seed, hasil GPU dan CPU boleh dipakai berdampingan.

In [ ]:
DATA_NAME = 'A1'                 # A1 = T=10; atau A1_T40, dst.
SCHEME    = 'within_event'       # within_event | leave_event_out
EPOCHS    = 15
SEEDS     = '42 7 123'
FUSIONS   = 'concat_glu'         # contoh banyak varian: 'concat_glu concat_fc sum attn lmf'

In [ ]:
%%bash -s "$ROOT" "$DRIVE_DIR" "$DATA_NAME" "$SCHEME" "$EPOCHS" "$SEEDS" "$FUSIONS"
cd "$1"
export TRAIN_PY=/content/venv/bin/python RESULT_DIR="$2/result"
DATA_NAME="$3" bash pipeline/run_grid.sh "$4" "$5" "$6" "$7"

## 7. Setelah selesai
- Salin folder `DRIVE_DIR/result` ke laptop dan gabungkan ke `work/result/`. Nama folder per run adalah hash dari parameter (termasuk path hasil), jadi tidak akan bertabrakan dengan run CPU lama.
- **Putuskan runtime** supaya compute unit tidak terus terpakai. Sel di bawah melakukannya; jalankan hanya setelah memastikan hasil sudah ada di Drive.

In [ ]:
from google.colab import runtime
runtime.unassign()